In [1]:
!pip install torch torchvision transformers huggingface-hub pillow tqdm numpy pycocoevalcap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.3/104.3 MB 15.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pycocoevalcap]0m [pycocoevalcap]

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [4]:
!pip install einops timm 


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [1]:
!pip install flash-attn==2.7.4.post1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.0/6.0 MB 1.6 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
  DEPRECATION: Building 'flash-attn' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'flash-attn'. Discussion can be found at https://github.com/pypa/pip/issues/6334
anceled

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
ERROR: Operation cancelled by user


In [1]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', 'transformers', '-y'])
subprocess.run([sys.executable, '-m', 'pip', 'install', 'transformers==4.38.0'])

print("✓ Done! NOW RESTART YOUR KERNEL")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 1.1 MB/s eta 0:00:0000:0100:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 1.7 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.1
    Uninstalling tokenizers-0.21.1:
      Successfully uninstalled tokenizers-0.21.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [transformers] [transformers]
✓ Done! NOW RESTART YOUR KERNEL



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [3]:
!pip show torch

Name: torch
Version: 2.7.0
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3-Clause
Location: /root/miniconda3/envs/py3.10/lib/python3.10/site-packages
Requires: filelock, fsspec, jinja2, networkx, nvidia-cublas-cu12, nvidia-cuda-cupti-cu12, nvidia-cuda-nvrtc-cu12, nvidia-cuda-runtime-cu12, nvidia-cudnn-cu12, nvidia-cufft-cu12, nvidia-cufile-cu12, nvidia-curand-cu12, nvidia-cusolver-cu12, nvidia-cusparse-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvjitlink-cu12, nvidia-nvtx-cu12, sympy, triton, typing-extensions
Required-by: flash_attn, timm, torchaudio, torchvision


In [5]:
import os
import shutil
from pathlib import Path
from huggingface_hub import snapshot_download, login
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

In [6]:
import json
import os
import argparse
from pathlib import Path
from typing import Dict, List, Tuple
from collections import defaultdict
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import numpy as np
import re

In [7]:
from huggingface_hub import login

try:
    from pycocoevalcap.bleu.bleu import Bleu
    from pycocoevalcap.meteor.meteor import Meteor
    from pycocoevalcap.rouge.rouge import Rouge
    from pycocoevalcap.cider.cider import Cider
    METRICS_AVAILABLE = True
except ImportError:
    print("Warning: pycocoevalcap not installed. Install with: pip install pycocoevalcap")
    METRICS_AVAILABLE = False

In [8]:
login()

In [9]:
def download_and_fix_falcon_model(model_id, local_dir='./model_checkpoints'):
    """
    Download Falcon model and fix the directory structure
    
    Args:
        model_id: HuggingFace model ID (e.g., 'TianHuiLab/Falcon-Single-Instruction-Large')
        local_dir: Where to save the fixed model
    
    Returns:
        Path to the fixed model directory
    """
    print("="*60)
    print("DOWNLOADING AND FIXING FALCON MODEL")
    print("="*60)
    
    # Extract model name
    model_name = model_id.split('/')[-1]
    download_path = Path(local_dir) / 'downloaded' / model_name
    fixed_path = Path(local_dir) / model_name
    
    # Check if already fixed
    if fixed_path.exists() and (fixed_path / 'config.json').exists():
        print(f"\n✓ Model already exists at: {fixed_path}")
        return str(fixed_path)
    
    # Step 1: Download from HuggingFace
    print(f"\n1. Downloading model from HuggingFace...")
    print(f"   This may take several minutes...")
    
    try:
        downloaded = snapshot_download(
            repo_id=model_id,
            local_dir=download_path,
            local_dir_use_symlinks=False
        )
        print(f"   ✓ Downloaded to: {downloaded}")
    except Exception as e:
        print(f"   ✗ Download failed: {e}")
        print("\n   Make sure you're logged in:")
        print("   from huggingface_hub import login")
        print("   login()")
        return None
    
    # Step 2: Find the actual model files (they're in a subdirectory)
    print(f"\n2. Looking for model files...")
    
    # Check if files are in root
    if (download_path / 'config.json').exists():
        print(f"   ✓ Files are in correct location")
        model_files_path = download_path
    else:
        # Files are in subdirectory
        nested_path = download_path / model_name
        if nested_path.exists() and (nested_path / 'config.json').exists():
            print(f"   ✓ Found files in: {nested_path}")
            model_files_path = nested_path
        else:
            print(f"   ✗ Could not find config.json")
            print(f"   Searched in:")
            print(f"   - {download_path}")
            print(f"   - {nested_path}")
            return None
    
    # Step 3: Move/copy files to fixed location
    print(f"\n3. Organizing files...")
    
    fixed_path.mkdir(parents=True, exist_ok=True)
    
    # Copy all files from source to destination
    for item in model_files_path.iterdir():
        dest = fixed_path / item.name
        if item.is_file():
            if not dest.exists():
                shutil.copy2(item, dest)
                print(f"   Copied: {item.name}")
        elif item.is_dir():
            if not dest.exists():
                shutil.copytree(item, dest)
                print(f"   Copied directory: {item.name}")
    
    print(f"\n✓ Model fixed and saved to: {fixed_path}")
    
    # Step 4: Verify
    required_files = ['config.json', 'pytorch_model.bin', 'preprocessor_config.json']
    missing = [f for f in required_files if not (fixed_path / f).exists()]
    
    if missing:
        print(f"\n⚠ Warning: Some files might be missing: {missing}")
        print(f"   Checking for alternative file names...")
        # Check for safetensors instead of pytorch_model.bin
        if 'pytorch_model.bin' in missing:
            safetensors = list(fixed_path.glob('*.safetensors'))
            if safetensors:
                print(f"   ✓ Found safetensors: {[f.name for f in safetensors]}")
                missing.remove('pytorch_model.bin')
    
    if not missing:
        print(f"\n✅ All required files present!")
    
    return str(fixed_path)


In [10]:
def load_falcon_local(model_path, device='cuda'):
    """
    Load Falcon model from local path
    
    Args:
        model_path: Local path to model directory
        device: 'cuda' or 'cpu'
    
    Returns:
        model, processor
    """
    print(f"\nLoading Falcon from: {model_path}")
    
    try:
        # Load processor
        print("Loading processor...")
        processor = AutoProcessor.from_pretrained(
            model_path,
            trust_remote_code=True,
            local_files_only=True
        )
        print("✓ Processor loaded")
        
        # Load model
        print("Loading model...")
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            trust_remote_code=True,
            torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
            local_files_only=True
        )
        model.to(device)
        model.eval()
        print(f"✓ Model loaded on {device}")
        
        return model, processor
        
    except Exception as e:
        print(f"✗ Loading failed: {e}")
        raise

In [11]:
def setup_falcon():
    """Complete setup: download, fix, and load Falcon model"""
    
    # Login to HuggingFace
    print("Step 1: Login to HuggingFace")
    try:
        login()
        print("✓ Logged in")
    except:
        print("⚠ Already logged in or login failed")
    
    # Download and fix model
    print("\nStep 2: Download and fix model structure")
    model_id = 'TianHuiLab/Falcon-Single-Instruction-Large'
    fixed_path = download_and_fix_falcon_model(model_id)
    
    if fixed_path is None:
        print("\n❌ Setup failed")
        return None, None, None
    
    # Load model
    print("\nStep 3: Load model")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model, processor = load_falcon_local(fixed_path, device)
    
    print("\n" + "="*60)
    print("✅ SETUP COMPLETE!")
    print("="*60)
    print(f"Model path: {fixed_path}")
    print(f"Device: {device}")
    
    return model, processor, fixed_path


In [12]:
# After restart
model, processor, model_path = setup_falcon()

Step 1: Login to HuggingFace


✓ Logged in

Step 2: Download and fix model structure
DOWNLOADING AND FIXING FALCON MODEL

✓ Model already exists at: model_checkpoints/Falcon-Single-Instruction-Large

Step 3: Load model

Loading Falcon from: model_checkpoints/Falcon-Single-Instruction-Large
Loading processor...


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


✓ Processor loaded
Loading model...


/root/miniconda3/envs/py3.10/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


✓ Model loaded on cuda

✅ SETUP COMPLETE!
Model path: model_checkpoints/Falcon-Single-Instruction-Large
Device: cuda


In [13]:
class Config:
    # Dataset paths
    IMAGES_DIR = 'VRSBench_val/Images_val'
    ANNOTATIONS_DIR = 'VRSBench_val/Annotations_val'
    
    # Model settings
    MODEL_NAME = 'TianHuiLab/Falcon-Single-Instruction-Large'  # Options:
    # 'TianHuiLab/Falcon-Single-Instruction-Large'
    # 'TianHuiLab/Falcon-Single-Instruction-Base'
    # 'TianHuiLab/Falcon-Multi-Instruction-Large'
    
    HF_TOKEN = None  # Set your HuggingFace token here, or None if already logged in
    
    # Output settings
    OUTPUT_DIR = './results'
    
    # Evaluation settings
    BATCH_SIZE = 1  # Keep at 1 for variable image sizes
    NUM_WORKERS = 4
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Tasks to evaluate
    TASKS = ['caption', 'grounding', 'vqa']  # Choose from: 'caption', 'grounding', 'vqa'
    
    # Generation parameters
    CAPTION_MAX_TOKENS = 512
    GROUNDING_MAX_TOKENS = 256
    VQA_MAX_TOKENS = 128
    NUM_BEAMS = 3



In [14]:
class VRSBenchDataset(Dataset):
    """Dataset class for VRSBench validation data"""
    
    def __init__(self, images_dir: str, annotations_dir: str, max_samples: int = None):
        self.images_dir = Path(images_dir)
        self.annotations_dir = Path(annotations_dir)
        
        # Load all annotation files
        self.annotations = []
        json_files = list(self.annotations_dir.glob("*.json"))
        
        if len(json_files) == 0:
            raise ValueError(f"No JSON files found in {annotations_dir}")
        
        for json_file in sorted(json_files):
            with open(json_file, 'r') as f:
                ann = json.load(f)
                ann['json_path'] = str(json_file)
                self.annotations.append(ann)
        
        # Limit samples if max_samples is specified
        if max_samples is not None:
            self.annotations = self.annotations[:max_samples]
        
        print(f"✓ Loaded {len(self.annotations)} samples from {annotations_dir}")
    
    def __len__(self):
        return len(self.annotations)
    
    def __getitem__(self, idx):
        ann = self.annotations[idx]
        image_path = self.images_dir / ann['image']
        
        # Load image
        try:
            image = Image.open(image_path).convert('RGB')
        except Exception as e:
            print(f"Error loading image {image_path}: {e}")
            image = None
        
        return {
            'image': image,
            'image_path': str(image_path),
            'annotation': ann
        }


# ============================================================================
# EVALUATOR CLASS
# ============================================================================
class VRSBenchEvaluator:
    """Evaluator for VRSBench tasks"""
    
    def __init__(self, model, processor, config):
        self.model = model
        self.processor = processor
        self.config = config
        self.device = config.DEVICE
        self.model.eval()
        
        # Detect model dtype
        self.model_dtype = next(model.parameters()).dtype
        
    def parse_box_from_response(self, response: str, image_width: int = 1000, image_height: int = 1000) -> List[float]:
        """Parse bounding box from Falcon's response.

        Supports integers and floats, pixel coords or normalized (0-1).
        Returns normalized [x1, y1, x2, y2].
        """
        if not response or not response.strip():
            print(f"Warning: Empty response when parsing box")
            return [0.4, 0.4, 0.6, 0.6]

        # Try several patterns including floats
        float_pat = r'([0-9]+(?:\.[0-9]+)?)'
        patterns = [
            # <box>X,Y,X,Y</box>
            rf'<box>{float_pat}\s*,\s*{float_pat}\s*,\s*{float_pat}\s*,\s*{float_pat}</box>',
            # [X, Y, X, Y]
            rf'\[{float_pat}\s*,\s*{float_pat}\s*,\s*{float_pat}\s*,\s*{float_pat}\]',
            # plain numbers separated by commas
            rf'^{float_pat}\s*,\s*{float_pat}\s*,\s*{float_pat}\s*,\s*{float_pat}$',
            # x1: 12 y1: 34 x2: 56 y2: 78
            rf'x1\s*[:=]\s*{float_pat}.*?y1\s*[:=]\s*{float_pat}.*?x2\s*[:=]\s*{float_pat}.*?y2\s*[:=]\s*{float_pat}',
            # x:12, y:34, w:56, h:78  (convert to x1,y1,x2,y2)
            rf'x\s*[:=]\s*{float_pat}.*?y\s*[:=]\s*{float_pat}.*?w\s*[:=]\s*{float_pat}.*?h\s*[:=]\s*{float_pat}',
            # any four floats anywhere
            rf'{float_pat}.*?{float_pat}.*?{float_pat}.*?{float_pat}'
        ]

        for pat in patterns:
            m = re.search(pat, response, flags=re.IGNORECASE | re.DOTALL)
            if m:
                groups = [g for g in m.groups() if g is not None]
                if len(groups) >= 4:
                    try:
                        x1, y1, x2, y2 = map(float, groups[:4])
                    except Exception:
                        continue

                    # If values are normalized (<=1), keep them
                    if all(0.0 <= v <= 1.0 for v in [x1, y1, x2, y2]):
                        return [x1, y1, x2, y2]

                    # If values look like pixel coords, convert to normalized
                    # We assume x values relate to image_width and y values to image_height
                    if max(x1, x2) > 1.0 or max(y1, y2) > 1.0:
                        nx1 = x1 / image_width
                        ny1 = y1 / image_height
                        nx2 = x2 / image_width
                        ny2 = y2 / image_height
                        return [nx1, ny1, nx2, ny2]

        # If no pattern matched, try extracting all floats and take the first 4
        floats = re.findall(float_pat, response)
        if len(floats) >= 4:
            vals = list(map(float, [f[0] if isinstance(f, tuple) else f for f in floats]))
            x1, y1, x2, y2 = vals[:4]
            if all(0.0 <= v <= 1.0 for v in [x1, y1, x2, y2]):
                return [x1, y1, x2, y2]
            if max(x1, x2) > 1.0 or max(y1, y2) > 1.0:
                return [x1 / image_width, y1 / image_height, x2 / image_width, y2 / image_height]

        print(f"Warning: Could not parse box from response: {response!r}")
        return [0.4, 0.4, 0.6, 0.6]
    
    def evaluate_caption(self, predictions: Dict, ground_truths: Dict) -> Dict:
        """Evaluate image captioning using standard metrics"""
        if not METRICS_AVAILABLE:
            print("Warning: Evaluation metrics not available. Install pycocoevalcap.")
            return {}
        
        gts = {img_id: [caption] for img_id, caption in ground_truths.items()}
        res = {img_id: [caption] for img_id, caption in predictions.items()}
        
        # Safely construct scorers; METEOR requires Java and may fail at instantiation
        scorers = []
        try:
            scorers.append((Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]))
        except Exception as e:
            print(f"Warning: BLEU scorer unavailable: {e}")
        
        try:
            # Meteor starts a Java subprocess; handle failure gracefully
            try:
                meteor = Meteor()
                scorers.append((meteor, "METEOR"))
            except Exception as me:
                print(f"Warning: METEOR unavailable (needs Java/METEOR jar): {me}")
        except Exception:
            pass
        
        try:
            scorers.append((Rouge(), "ROUGE_L"))
        except Exception as e:
            print(f"Warning: ROUGE scorer unavailable: {e}")
        
        try:
            scorers.append((Cider(), "CIDEr"))
        except Exception as e:
            print(f"Warning: CIDEr scorer unavailable: {e}")
        
        if not scorers:
            print("No scorers available. Skipping caption evaluation.")
            return {}
        
        scores = {}
        for scorer, method in scorers:
            try:
                score, _ = scorer.compute_score(gts, res)
                if isinstance(method, list):
                    for m, s in zip(method, score):
                        scores[m] = s
                else:
                    scores[method] = score
            except Exception as e:
                print(f"Error computing {method}: {e}")
        
        return scores
    
    def calculate_iou(self, box1: List[float], box2: List[float]) -> float:
        """Calculate IoU between two bounding boxes"""
        x1_min, y1_min, x1_max, y1_max = box1
        x2_min, y2_min, x2_max, y2_max = box2
        
        inter_xmin = max(x1_min, x2_min)
        inter_ymin = max(y1_min, y2_min)
        inter_xmax = min(x1_max, x2_max)
        inter_ymax = min(y1_max, y2_max)
        
        if inter_xmax <= inter_xmin or inter_ymax <= inter_ymin:
            return 0.0
        
        inter_area = (inter_xmax - inter_xmin) * (inter_ymax - inter_ymin)
        box1_area = (x1_max - x1_min) * (y1_max - y1_min)
        box2_area = (x2_max - x2_min) * (y2_max - y2_min)
        union_area = box1_area + box2_area - inter_area
        
        return inter_area / union_area if union_area > 0 else 0.0
    
    def evaluate_grounding(self, predictions: List[Dict], ground_truths: List[Dict],
                          iou_thresholds: List[float] = [0.5]) -> Dict:
        """Evaluate visual grounding performance"""
        pred_by_image = defaultdict(list)
        gt_by_image = defaultdict(list)
        
        for pred in predictions:
            pred_by_image[pred['image_id']].append(pred)
        
        for gt in ground_truths:
            gt_by_image[gt['image_id']].append(gt)
        
        results = {f'Acc@{t}': [] for t in iou_thresholds}
        
        for image_id in gt_by_image.keys():
            gt_boxes = gt_by_image[image_id]
            pred_boxes = pred_by_image.get(image_id, [])
            
            for gt in gt_boxes:
                gt_obj_id = gt['obj_id']
                gt_box = gt['box']
                
                pred_box = None
                for pred in pred_boxes:
                    if pred['obj_id'] == gt_obj_id:
                        pred_box = pred['box']
                        break
                
                if pred_box is not None:
                    iou = self.calculate_iou(pred_box, gt_box)
                    for threshold in iou_thresholds:
                        results[f'Acc@{threshold}'].append(1.0 if iou >= threshold else 0.0)
                else:
                    for threshold in iou_thresholds:
                        results[f'Acc@{threshold}'].append(0.0)
        
        return {k: np.mean(v) * 100 if v else 0.0 for k, v in results.items()}
    
    def evaluate_vqa(self, predictions: Dict, ground_truths: Dict) -> Dict:
        """Evaluate VQA performance"""
        correct = 0
        total = 0
        type_correct = defaultdict(int)
        type_total = defaultdict(int)
        
        for key, gt_answer in ground_truths.items():
            pred_answer = predictions.get(key, "")
            
            gt_norm = self.normalize_answer(gt_answer['answer'])
            pred_norm = self.normalize_answer(pred_answer)
            
            is_correct = (gt_norm == pred_norm)
            
            correct += int(is_correct)
            total += 1
            
            q_type = gt_answer.get('type', 'unknown')
            type_correct[q_type] += int(is_correct)
            type_total[q_type] += 1
        
        results = {
            'overall_accuracy': (correct / total * 100) if total > 0 else 0.0,
            'total_questions': total
        }
        
        for q_type in type_total:
            results[f'accuracy_{q_type}'] = (
                type_correct[q_type] / type_total[q_type] * 100
                if type_total[q_type] > 0 else 0.0
            )
        
        return results
    
    @staticmethod
    def normalize_answer(answer: str) -> str:
        """Normalize answer string for comparison"""
        answer = str(answer).lower().strip()
        answer = re.sub(r'[^\w\s]', '', answer)
        answer = ' '.join(answer.split())
        return answer
    
    def run_inference_caption(self, dataloader: DataLoader) -> Tuple[Dict, Dict]:
        """Run inference for image captioning task"""
        predictions = {}
        ground_truths = {}
        
        with torch.no_grad():
            for batch in tqdm(dataloader, desc="Captioning"):
                images = batch['image']
                annotations = batch['annotation']
                
                for i, img in enumerate(images):
                    if img is None:
                        continue
                    
                    ann = annotations[i]
                    image_id = ann['image']
                    
                    try:
                        prompt = "Describe the image in detail."
                        
                        inputs = self.processor(
                            text=prompt, 
                            images=img, 
                            return_tensors="pt"
                        ).to(self.device)
                        
                        # Convert to model dtype
                        inputs = {k: v.to(self.model_dtype) if v.dtype in [torch.float32, torch.float64] else v 
                                 for k, v in inputs.items()}
                        
                        generated_ids = self.model.generate(
                            **inputs,
                            max_new_tokens=self.config.CAPTION_MAX_TOKENS,
                            num_beams=self.config.NUM_BEAMS,
                            do_sample=False
                        )
                        
                        pred_caption = self.processor.batch_decode(
                            generated_ids, 
                            skip_special_tokens=True
                        )[0]
                        
                        if prompt in pred_caption:
                            pred_caption = pred_caption.replace(prompt, "").strip()
                        
                        predictions[image_id] = pred_caption
                        ground_truths[image_id] = ann['caption']
                        
                    except Exception as e:
                        print(f"Error processing {image_id}: {e}")
                        predictions[image_id] = ""
                        ground_truths[image_id] = ann['caption']
        
        return predictions, ground_truths
    
    def run_inference_grounding(self, dataloader: DataLoader) -> Tuple[List[Dict], List[Dict]]:
        """Run inference for visual grounding task"""
        predictions = []
        ground_truths = []
        
        with torch.no_grad():
            for batch in tqdm(dataloader, desc="Visual Grounding"):
                images = batch['image']
                annotations = batch['annotation']
                
                for i, img in enumerate(images):
                    if img is None:
                        continue
                    
                    ann = annotations[i]
                    image_id = ann['image']
                    img_width, img_height = img.size
                    
                    for obj in ann['objects']:
                        try:
                            # Build a stronger prompt using object class + few-shot examples
                            prompt = (
                                f"Object class: {obj.get('obj_cls','object')}\n"
                                f"Description: {obj['referring_sentence']}\n\n"
                                "You MUST respond ONLY with a single bounding box in this exact format:\n"
                                "<box>x1,y1,x2,y2</box>\n"
                                "- Coordinates should be integers in PIXEL coordinates (top-left x,y and bottom-right x,y).\n"
                                "- Do not include any other text or explanation.\n\n"
                                "Example 1:\n"
                                "Input: The red car parked at the bottom-left corner.\n"
                                "Output: <box>30,420,150,520</box>\n\n"
                                "Example 2:\n"
                                "Input: A small circular fountain near the center.\n"
                                "Output: <box>480,300,540,360</box>\n\n"
                                "Now provide the box for the input above (Object class + Description)."
                            )

                            # Helper to generate and parse once
                            def gen_and_parse(inp_prompt, do_sample=False, num_beams=None, top_p=0.9, temperature=0.8):
                                inputs = self.processor(text=inp_prompt, images=img, return_tensors="pt").to(self.device)
                                inputs = {k: v.to(self.model_dtype) if hasattr(v, 'dtype') and v.dtype in [torch.float32, torch.float64] else v for k, v in inputs.items()}
                                gen_kwargs = {
                                    'max_new_tokens': self.config.GROUNDING_MAX_TOKENS,
                                    'do_sample': do_sample,
                                }
                                if num_beams is not None:
                                    gen_kwargs['num_beams'] = num_beams
                                if do_sample:
                                    gen_kwargs['top_p'] = top_p
                                    gen_kwargs['temperature'] = temperature

                                generated_ids = self.model.generate(**inputs, **gen_kwargs)
                                response = self.processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
                                parsed_box = self.parse_box_from_response(response, img_width, img_height)
                                return response, parsed_box

                            # Attempt 1: deterministic beam search
                            resp1, box1 = gen_and_parse(prompt, do_sample=False, num_beams=self.config.NUM_BEAMS)
                            if box1 is None or box1 == [0.4,0.4,0.6,0.6]:
                                # Print raw response for debugging
                                print(f"GROUNDING ATTEMPT 1 RAW RESPONSE: {resp1!r}")
                                # Attempt 2: sampled generation
                                resp2, box2 = gen_and_parse(prompt, do_sample=True, num_beams=1, top_p=0.9, temperature=0.7)
                                if box2 is None or box2 == [0.4,0.4,0.6,0.6]:
                                    print(f"GROUNDING ATTEMPT 2 RAW RESPONSE: {resp2!r}")
                                    # Attempt 3: fallback asking for normalized coordinates
                                    fallback = (
                                        f"Object class: {obj.get('obj_cls','object')}\n"
                                        f"Description: {obj['referring_sentence']}\n\n"
                                        "If you cannot provide pixel coordinates, return normalized coordinates between 0 and 1 in this format:\n"
                                        "<box>x1,y1,x2,y2</box>\n"
                                        "Example: <box>0.65,0.58,0.69,0.63</box>\n"
                                        "Return the box now and nothing else."
                                    )
                                    resp3, box3 = gen_and_parse(fallback, do_sample=True, num_beams=1, top_p=0.9, temperature=0.7)
                                    print(f"GROUNDING ATTEMPT 3 RAW RESPONSE: {resp3!r}")
                                    final_box = box3 if box3 and box3 != [0.4,0.4,0.6,0.6] else [0.4,0.4,0.6,0.6]
                                else:
                                    final_box = box2
                            else:
                                final_box = box1

                            predictions.append({
                                'image_id': image_id,
                                'obj_id': obj['obj_id'],
                                'box': final_box
                            })

                        except Exception as e:
                            print(f"Error processing object {obj['obj_id']} in {image_id}: {e}")
                            predictions.append({
                                'image_id': image_id,
                                'obj_id': obj['obj_id'],
                                'box': [0.4, 0.4, 0.6, 0.6]
                            })

                        ground_truths.append({
                            'image_id': image_id,
                            'obj_id': obj['obj_id'],
                            'box': obj['obj_coord']
                        })
        
        return predictions, ground_truths
    
    def run_inference_vqa(self, dataloader: DataLoader) -> Tuple[Dict, Dict]:
        """Run inference for VQA task"""
        predictions = {}
        ground_truths = {}
        
        def extract_short_answer(text: str) -> str:
            """Extract a single-word or numeric answer from the model output."""
            if not text:
                return ""
            s = str(text).strip()
            # remove common prefixes
            s = re.sub(r'^(answer|ans|a[:]?)\s*[:\-]*\s*', '', s, flags=re.I)
            # take the first non-empty line
            lines = [l.strip() for l in s.splitlines() if l.strip()]
            if lines:
                s = lines[0]
            # remove any surrounding explanation like "Answer: ..." again
            s = re.sub(r'^(answer|ans|the answer is)[:\.\-\s]*', '', s, flags=re.I)
            # find first token that looks like a number or word (allow hyphen/apostrophe)
            m = re.search(r"([0-9]+(?:\.[0-9]+)?)|([A-Za-z][A-Za-z'\-]+)", s)
            if m:
                token = m.group(0)
                return token.lower()
            # fallback: split on whitespace and take first token
            parts = re.sub(r'[\W_]+', ' ', s).split()
            return parts[0].lower() if parts else ""
        
        with torch.no_grad():
            for batch in tqdm(dataloader, desc="VQA"):
                images = batch['image']
                annotations = batch['annotation']
                
                for i, img in enumerate(images):
                    if img is None:
                        continue
                    
                    ann = annotations[i]
                    image_id = ann['image']
                    
                    for qa in ann['qa_pairs']:
                        try:
                            question = qa['question']
                            key = (image_id, qa['ques_id'])
                            
                            inputs = self.processor(
                                text=question,
                                images=img,
                                return_tensors="pt"
                            ).to(self.device)
                            
                            # Convert to model dtype
                            inputs = {k: v.to(self.model_dtype) if hasattr(v, 'dtype') and v.dtype in [torch.float32, torch.float64] else v 
                                     for k, v in inputs.items()}
                            
                            generated_ids = self.model.generate(
                                **inputs,
                                max_new_tokens=self.config.VQA_MAX_TOKENS,
                                num_beams=self.config.NUM_BEAMS,
                                do_sample=False
                            )
                            
                            pred_answer_raw = self.processor.batch_decode(
                                generated_ids,
                                skip_special_tokens=True
                            )[0]
                            
                            # Remove the question if model echoes it
                            if question.strip().lower() in pred_answer_raw.strip().lower():
                                pred_answer_raw = pred_answer_raw.replace(question, '').strip()

                            # Extract short answer (one word/number)
                            pred_answer = extract_short_answer(pred_answer_raw)

                            predictions[key] = pred_answer
                            ground_truths[key] = {
                                'answer': qa['answer'],
                                'type': qa['type']
                            }
                            
                        except Exception as e:
                            print(f"Error processing QA {qa['ques_id']} in {image_id}: {e}")
                            predictions[key] = ""
                            ground_truths[key] = {
                                'answer': qa['answer'],
                                'type': qa['type']
                            }
        
        return predictions, ground_truths

In [15]:
def run_evaluation(model, processor, config=None, max_samples=None):
    """
    Main evaluation function
    
    Args:
        model: Pre-loaded model
        processor: Pre-loaded processor
        config: Config object with settings (uses default Config if None)
        max_samples: Max number of samples to evaluate (None = all samples)
    """
    if config is None:
        config = Config()
    
    # Create output directory
    os.makedirs(config.OUTPUT_DIR, exist_ok=True)
    
    print(f"\n{'='*60}")
    print(f"Using pre-loaded model")
    print(f"{'='*60}")
    print(f"✓ Model loaded on {config.DEVICE}")
    if max_samples:
        print(f"✓ Evaluating on {max_samples} samples")
    
    # Create dataset and dataloader
    print(f"\n{'='*60}")
    print(f"Loading dataset")
    print(f"{'='*60}")
    print(f"Images: {config.IMAGES_DIR}")
    print(f"Annotations: {config.ANNOTATIONS_DIR}")
    
    try:
        dataset = VRSBenchDataset(config.IMAGES_DIR, config.ANNOTATIONS_DIR, max_samples=max_samples)
        dataloader = DataLoader(
            dataset,
            batch_size=config.BATCH_SIZE,
            shuffle=False,
            num_workers=config.NUM_WORKERS,
            collate_fn=lambda x: {
                'image': [item['image'] for item in x],
                'annotation': [item['annotation'] for item in x]
            }
        )
    except Exception as e:
        print(f"✗ Error loading dataset: {e}")
        return None
    
    # Create evaluator
    evaluator = VRSBenchEvaluator(model, processor, config)
    
    results = {}
    
    # Evaluate each task
    if 'caption' in config.TASKS:
        print(f"\n{'='*60}")
        print("EVALUATING IMAGE CAPTIONING")
        print(f"{'='*60}")
        
        predictions, ground_truths = evaluator.run_inference_caption(dataloader)
        
        pred_path = os.path.join(config.OUTPUT_DIR, 'caption_predictions.json')
        with open(pred_path, 'w') as f:
            json.dump(predictions, f, indent=2)
        print(f"✓ Predictions saved to {pred_path}")
        
        caption_scores = evaluator.evaluate_caption(predictions, ground_truths)
        results['caption'] = caption_scores
        
        print("\n📊 Captioning Results:")
        for metric, score in caption_scores.items():
            print(f"  {metric:12s}: {score:.4f}")
    
    if 'grounding' in config.TASKS:
        print(f"\n{'='*60}")
        print("EVALUATING VISUAL GROUNDING")
        print(f"{'='*60}")
        
        predictions, ground_truths = evaluator.run_inference_grounding(dataloader)
        
        pred_path = os.path.join(config.OUTPUT_DIR, 'grounding_predictions.json')
        with open(pred_path, 'w') as f:
            json.dump(predictions, f, indent=2)
        print(f"✓ Predictions saved to {pred_path}")
        
        grounding_scores = evaluator.evaluate_grounding(
            predictions, ground_truths, 
            iou_thresholds=[0.5, 0.75]
        )
        results['grounding'] = grounding_scores
        
        print("\n📊 Visual Grounding Results:")
        for metric, score in grounding_scores.items():
            print(f"  {metric:12s}: {score:.2f}%")
    
    if 'vqa' in config.TASKS:
        print(f"\n{'='*60}")
        print("EVALUATING VISUAL QUESTION ANSWERING")
        print(f"{'='*60}")
        
        predictions, ground_truths = evaluator.run_inference_vqa(dataloader)
        
        pred_list = [{'image_id': k[0], 'ques_id': k[1], 'answer': v} for k, v in predictions.items()]
        pred_path = os.path.join(config.OUTPUT_DIR, 'vqa_predictions.json')
        with open(pred_path, 'w') as f:
            json.dump(pred_list, f, indent=2)
        print(f"✓ Predictions saved to {pred_path}")
        
        vqa_scores = evaluator.evaluate_vqa(predictions, ground_truths)
        results['vqa'] = vqa_scores
        
        print("\n📊 VQA Results:")
        for metric, score in vqa_scores.items():
            if 'accuracy' in metric:
                print(f"  {metric:30s}: {score:.2f}%")
            else:
                print(f"  {metric:30s}: {score}")
    
    # Save all results
    results_path = os.path.join(config.OUTPUT_DIR, 'evaluation_results.json')
    with open(results_path, 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"\n{'='*60}")
    print(f"✓ EVALUATION COMPLETE!")
    print(f"✓ Results saved to {results_path}")
    print(f"{'='*60}\n")
    
    return results



In [16]:
config = Config()
# Run only caption and vqa (skip grounding)
config.TASKS = ['vqa']
#results = run_evaluation(model, processor, config, max_samples=500)


In [24]:
# Patch: improved normalize_answer and quick re-eval/demo
import re
from collections import defaultdict

# Numeric words mapping
NUM_WORDS = {
    'zero':'0','one':'1','two':'2','three':'3','four':'4','five':'5','six':'6','seven':'7','eight':'8','nine':'9',
    'ten':'10','eleven':'11','twelve':'12'
}

# Synonyms mapping (extend as needed)
SYNONYMS = {
    'windmill': 'wind turbine',
    'wind-turbine': 'wind turbine',
    'overpass': 'bridge',
    'trainstation': 'train station',
    'train station': 'train station',
    'plane': 'airplane',
    'car': 'vehicle',
    'truck': 'vehicle',
    'bridges': 'bridge',
    'ground track field': 'ground track field',
}


def normalize_answer_improved(answer: str) -> str:
    """Robust normalization: extract concise answer from verbose text.
    Returns a short, lowercased string (1-3 tokens) suitable for equality checks.
    """
    if answer is None:
        return ''
    s = str(answer).strip()
    if not s:
        return ''

    # Replace newlines and HTML-like tags
    s = s.replace('\n', ' ').replace('\r', ' ')
    s = re.sub(r'<[^>]+>', ' ', s)

    # Remove leading filler phrases often present in GTs
    s = re.sub(r'(?i)^(the image (is|shows|contains)|this image (is|shows)|the primary .*? is|the main .*? is|the object .*? is|there (is|are)|the answer (is|:)|answer[:\-])\s*', '', s)

    # If phrase contains 'is <something>' capture the part after the first ' is ' (useful for long sentences)
    m = re.search(r'(?i)\b(?:is|are|was|were|called)\s+(.*)', s)
    if m:
        s = m.group(1)

    # Keep only letters/numbers and spaces
    s = re.sub(r"[^A-Za-z0-9\s]", ' ', s)
    s = s.lower()
    tokens = [t for t in s.split() if t]
    if not tokens:
        return ''

    # Convert small number-words to digits
    for i, t in enumerate(tokens):
        if t in NUM_WORDS:
            tokens[i] = NUM_WORDS[t]

    # Join first up to 3 tokens as the canonical short answer
    cand = ' '.join(tokens[:3])

    # Map synonyms exactly
    if cand in SYNONYMS:
        cand = SYNONYMS[cand]
    else:
        # also try single-token synonym mapping
        parts = cand.split()
        if parts and parts[0] in SYNONYMS:
            cand = SYNONYMS[parts[0]]

    return cand

# Monkey-patch the evaluator's normalize_answer (so other cells use the new version)
try:
    VRSBenchEvaluator.normalize_answer = staticmethod(normalize_answer_improved)
    print('Patched VRSBenchEvaluator.normalize_answer with improved normalizer')
except Exception as e:
    print('Could not patch VRSBenchEvaluator.normalize_answer:', e)

# Demo: normalize the example GT strings you posted earlier
examples = [
    ('05863_0000.png', 1, 'Windmill'),
    ('05863_0000.png', 2, 'Terrain'),
    ('05864_0000.png', 1, 'bridges'),
    ('05866_0000.png', 1, 'overpass'),
    ('05867_0000.png', 1, 'overpass'),
    ('05868_0000.png', 1, 'windmill'),
    ('05869_0000.png', 3, 'ground track field'),
    ('05871_0000.png', 2, 'train station'),
    ('05875_0000.png', 1, 'train station'),
    ('05875_0000.png', 2, 'directional lanes'),
]
print('\nNormalized examples:')
for img, qid, raw in examples:
    print(f"  {img}, QID {qid}: raw={raw!r} -> norm={normalize_answer_improved(raw)!r}")

# If ground_truths and ext_predictions exist in the current notebook environment, re-run eval
if 'ground_truths' in globals() and 'ext_predictions' in globals():
    print('\nRe-running VQA evaluation with the improved normalizer...')
    evaluator = VRSBenchEvaluator(model, processor, config)
    try:
        scores = evaluator.evaluate_vqa(ext_predictions, ground_truths)
        print('\nCategory-wise VQA Results (after patch):')
        for metric, val in scores.items():
            if 'accuracy' in metric:
                print(f"  {metric:30s}: {val:.2f}%")
            else:
                print(f"  {metric:30s}: {val}")
    except Exception as e:
        print('Evaluation failed after patch:', e)
else:
    print('\nNote: ground_truths and ext_predictions not found in the current notebook state.\nIf you want to re-evaluate now, run the evaluation cell (that builds ground_truths and ext_predictions) again — it will use the patched normalizer.')


Patched VRSBenchEvaluator.normalize_answer with improved normalizer

Normalized examples:
  05863_0000.png, QID 1: raw='Windmill' -> norm='wind turbine'
  05863_0000.png, QID 2: raw='Terrain' -> norm='terrain'
  05864_0000.png, QID 1: raw='bridges' -> norm='bridge'
  05866_0000.png, QID 1: raw='overpass' -> norm='bridge'
  05867_0000.png, QID 1: raw='overpass' -> norm='bridge'
  05868_0000.png, QID 1: raw='windmill' -> norm='wind turbine'
  05869_0000.png, QID 3: raw='ground track field' -> norm='ground track field'
  05871_0000.png, QID 2: raw='train station' -> norm='train station'
  05875_0000.png, QID 1: raw='train station' -> norm='train station'
  05875_0000.png, QID 2: raw='directional lanes' -> norm='directional lanes'

Re-running VQA evaluation with the improved normalizer...
Evaluation failed after patch: 'answer'


In [26]:
# --- START CELL: improved normalization + category-wise VQA eval (uses per-image JSONs) ---
import re
import json
import os
from collections import defaultdict

# Choose deterministic first-N samples (set to 500)
MAX_SAMPLES = 500

# Path to external predictions (update if needed)
external_predictions_file = r'c:\path\to\your\external_predictions.jsonl'  # <<-- update

# --- Helper: extract short answer from verbose text ---
def extract_short_answer(text: str) -> str:
    if text is None:
        return ""
    s = str(text).strip()
    if not s:
        return ""
    # If sentence contains "is/are/answer: ..." try to capture what's after it
    m = re.search(r'\b(?:is|are|was|were|answer\s*[:\-]|the answer is|it is|this is|there is|there are)\s+(?:an?\s+|the\s+)?([^.;\n]+)', s, flags=re.I)
    if m:
        cand = m.group(1).strip()
    else:
        # fallback: take first meaningful token or word-phrase (allow two words)
        m2 = re.search(r'([0-9]+(?:\.[0-9]+)?)|([A-Za-z][A-Za-z\'\-/]+(?:\s+[A-Za-z\'\-/]+)?)', s)
        cand = m2.group(0) if m2 else s
    # split on punctuation or common trailing phrases
    cand = re.split(r'[.,;()\\-\\n]', cand)[0].strip()
    return cand

# Map numeric words to digits for easier comparison
NUM_WORDS = {
    'zero': '0','one':'1','two':'2','three':'3','four':'4','five':'5','six':'6','seven':'7','eight':'8','nine':'9',
    'ten':'10','eleven':'11','twelve':'12'
}

# Common synonym normalization (expand as needed)
SYNONYMS = {
    'windmill': 'wind turbine',
    'wind-turbine': 'wind turbine',
    'overpass': 'bridge',
    'trainstation': 'train station',
    'train station': 'train station',
    'basketball court': 'basketball court',
    'baseball diamond': 'baseball field',
    'plane': 'airplane',
    'car': 'vehicle',
    'truck': 'vehicle',
}

def normalize_answer_for_eval(ans: str) -> str:
    s = extract_short_answer(ans)
    s = s.lower().strip()
    # replace numeric words
    for w, d in NUM_WORDS.items():
        s = re.sub(r'\b' + re.escape(w) + r'\b', d, s)
    # collapse punctuation to spaces, but keep hyphens removed
    s = re.sub(r'[^a-z0-9\s]', ' ', s)
    s = ' '.join(s.split())
    # map synonyms
    for k, v in SYNONYMS.items():
        if s == k or s == k.replace(' ', ''):
            s = v
            break
    return s

# --- Load ground-truths using VRSBenchDataset (per-image JSONs) ---
dataset = VRSBenchDataset(config.IMAGES_DIR, config.ANNOTATIONS_DIR, max_samples=MAX_SAMPLES)
ground_truths = {}
for idx in range(len(dataset)):
    ann = dataset[idx]['annotation']
    image_id = ann.get('image')
    for qa in ann.get('qa_pairs', []):
        qid = qa.get('ques_id') or qa.get('qid')
        if image_id is None or qid is None:
            continue
        key = (image_id, qid)
        ground_truths[key] = {'raw_answer': qa.get('answer', ''), 'type': qa.get('type', 'unknown')}

print(f"Loaded {len(ground_truths)} GT QA pairs (first {MAX_SAMPLES} images)")


def load_flexible_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        text = f.read()
    # try list / object
    try:
        obj = json.loads(text)
        return obj if isinstance(obj, list) else [obj]
    except Exception:
        pass
    # try JSONL
    out = []
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    ok = True
    for ln in lines:
        try:
            out.append(json.loads(ln))
        except Exception:
            ok = False
            break
    if ok and out:
        return out
    # fallback concatenated JSON objects
    objs = []
    buf = ''
    depth = 0; in_str = False; esc = False
    for ch in text:
        buf += ch
        if ch == '"' and not esc:
            in_str = not in_str
        if ch == '\\' and not esc:
            esc = True
            continue
        else:
            esc = False
        if not in_str:
            if ch == '{': depth += 1
            elif ch == '}': depth -= 1
        if depth == 0 and buf.strip():
            try:
                objs.append(json.loads(buf.strip()))
            except Exception:
                pass
            buf = ''
    if objs:
        return objs
    raise ValueError(f"Could not parse JSON from: {path}")

if not os.path.exists(external_predictions_file):
    print(f"External predictions file not found at: {external_predictions_file}")
    ext_predictions = {}
else:
    preds_list = load_flexible_json(external_predictions_file)
    ext_predictions = {}
    for it in preds_list:
        image_id = it.get('image_id') or it.get('image') or it.get('img_id') or it.get('image_name')
        ques_id = it.get('ques_id') or it.get('question_id') or it.get('qid')
        answer = it.get('answer') or it.get('pred') or it.get('response') or it.get('prediction') or ''
        if image_id is None or ques_id is None:
            continue
        key = (image_id, ques_id)
        ext_predictions[key] = str(answer)
    print(f"Loaded {len(ext_predictions)} external predictions")

# --- Normalize both GT and predictions for fair comparison ---
norm_gt = {}
for k, v in ground_truths.items():
    norm_gt[k] = {'answer': normalize_answer_for_eval(v['raw_answer']), 'type': v['type'], 'raw': v['raw_answer']}

norm_pred = {}
for k, v in ext_predictions.items():
    norm_pred[k] = normalize_answer_for_eval(v)

# Filter predictions to the selected GT keys (only evaluate those)
filtered_preds = {k: norm_pred.get(k, '') for k in norm_gt.keys()}

# --- Compute category-wise accuracies manually (robust) ---
type_correct = defaultdict(int)
type_total = defaultdict(int)
total = 0
correct = 0
mismatches = defaultdict(list)

for k, g in norm_gt.items():
    gt_ans = g['answer']
    qtype = g['type']
    pred_ans = filtered_preds.get(k, '')
    total += 1
    type_total[qtype] += 1
    if gt_ans and pred_ans and gt_ans == pred_ans:
        correct += 1
        type_correct[qtype] += 1
    else:
        mismatches[qtype].append({'image': k[0], 'ques_id': k[1], 'gt_raw': g['raw'], 'gt_norm': gt_ans, 'pred_raw': ext_predictions.get(k, ''), 'pred_norm': pred_ans})

# Print results
overall_acc = (correct / total * 100) if total else 0.0
print("\nCategory-wise VQA Results (improved normalization):")
print(f"  overall_accuracy              : {overall_acc:.2f}%")
print(f"  total_questions               : {total}")
for qtype in sorted(type_total.keys()):
    acc = (type_correct[qtype] / type_total[qtype] * 100) if type_total[qtype] > 0 else 0.0
    print(f"  accuracy_{qtype:20s}: {acc:.2f}% (n={type_total[qtype]})")

# Show some mismatch examples per category
for qtype, items in mismatches.items():
    if not items:
        continue
    print(f"\nExamples incorrect for '{qtype}' (showing up to 10):")
    for ex in items[:10]:
        print(f"  Image: {ex['image']}, QID: {ex['ques_id']}")
        print(f"    GT raw : {ex['gt_raw']!r}")
        print(f"    GT norm: {ex['gt_norm']!r}")
        print(f"    PR raw : {ex['pred_raw']!r}")
        print(f"    PR norm: {ex['pred_norm']!r}")

# Save summary
os.makedirs(config.OUTPUT_DIR, exist_ok=True)
with open(os.path.join(config.OUTPUT_DIR, 'external_vqa_eval_normalized_summary.json'), 'w', encoding='utf-8') as f:
    json.dump({'overall_accuracy': overall_acc, 'total_questions': total, 'type_counts': {t: type_total[t] for t in type_total}, 'type_correct': {t: type_correct[t] for t in type_correct}}, f, indent=2, ensure_ascii=False)
with open(os.path.join(config.OUTPUT_DIR, 'external_vqa_mismatches_normalized.json'), 'w', encoding='utf-8') as f:
    json.dump(mismatches, f, indent=2, ensure_ascii=False)

print(f"\nSaved normalized summary and mismatches to {config.OUTPUT_DIR}")


JSONDecodeError: Extra data: line 2 column 1 (char 1291)